# Deposit Attrition EDA — v10c · money still on the book

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v10b answered its own question the wrong way round. The explanation offered for why
`payment_only` reached more dollars — that it was blind to client size — was falsified by its own
α sweep. A size artefact would grow with α and vanish at α = 0. Instead the advantage was
**largest at α = 0 (3.5×)**, where balance plays no part in the ranking, and shrank to 1.1× at α = 1.

What the α = 0 queue actually showed:

| set | departing clients reached | precision | $ per client |
|---|---|---|---|
| deposit_only | 2,061 | 58% | **$145** |
| all_features | 2,211 | 51% | $25k |
| payment_only | 1,249 | 30% | **$155k** |

The deposit model finds the most attriters at the highest precision, and they are **empty**. It
earns its AUC by predicting the closure of accounts whose money has already gone. **Deposit
signals see attrition after the money leaves; payment signals catch it while it is still on the
book.** This notebook tests that directly, and fixes five defects of mine.

| § | |
|---|---|
| 1 | Self-checks run against the **actual sampling design** — v10b's check sampled on the label alone, passed, and the real design still failed |
| 2 | **What α is**, with a worked example |
| 3 | Data. The training frame carries its own inclusion probability; an audit **measures** the v10b sampling defect |
| 4 | Inverse-probability-weighted fit and weighted isotonic, on two risk sets: the full book, and **money still here** |
| 5 | Calibration |
| 6 | **When does each queue arrive?** Share of reached attriters already empty |
| 7 | AUC with and without the empty accounts |
| 8 | Savings where the money still is; α = 1 reference; ranking on current against normal balance |
| 9 | Efficient frontier — how many calls |
| 10 | Permutation control, 20 per fold, `deposit_only` |
| 11 | Pool with `B_bal_exit`; lead ladder per client |
| 12 | **Scorecard** — the predictions below, checked mechanically |

### Defects fixed

1. **Calibration was still ~2× low, and the cause was the sampling design, not the arithmetic.**
   v10b kept a training row whole if the client had an A, D *or* B event within 12 months, so a
   large share of *negatives* were certain inclusions. The King & Zeng offset and the isotonic
   weights both assumed every negative had a 15% chance. The same rows biased the coefficients:
   B clients — drained, never closed — sat among the A-negatives at roughly six times their
   population weight. Fixed by inverse-probability weights computed from the very expression that
   did the filtering.
2. **No chart ever rendered.** Every `<Figure size … with 1 Axes>` in v10 and v10b was a figure
   that failed to display, because I forced the Agg backend, which disconnects Jupyter's image
   output. The PNGs were saved to disk and can still be opened. Charts are now displayed as PNG
   bytes, which works under any backend.
3. **The frontier filter was inverted.** It discarded cheaper points holding less money instead of
   costlier points holding no more, so only K = 5,000 ever survived. The "collapse" was not a
   finding.
4. **§9 crashed** on `PB.mean`, which is the DataFrame method, not the column.
5. **The fitter could diverge silently** — found while testing this notebook. The IRLS used since
   v7 takes full Newton steps with no step control. On a synthetic panel it drove coefficients past
   200,000 and saturated every prediction, which scores an AUC of exactly 0.5000. v10b's AUCs of
   0.72–0.80 suggest it converged on the real data, but nothing checked. Replaced with a damped
   Newton that can only improve the objective, and **every fit now reports convergence** (§4).

### Pre-registered predictions — checked in §12

| # | prediction | confidence |
|---|---|---|
| P1 | The audit implies a v10b miscalibration factor of **1.7–2.5×** (v10b observed 2.08) | ~75% |
| P2 | IPW + isotonic top-decile ratio lands in **0.80–1.25** for all four sets | ~70% |
| P3 | `deposit_only`, α = 0: **over 80%** of the departing clients it reaches are already drained — below 10% of their normal balance | ~75% |
| P4 | `payment_only`, α = 0: **under 50%** already drained | ~65% |
| P5 | `deposit_only` AUC falls by **≥ 0.05** when scored only where money is still on the book | ~65% |
| P6 | `payment_only` AUC falls by **< 0.03** on the same subset | ~55% |
| P7 | On the money-still-here risk set at α = 1, `all_features` reaches **at least** `payment_only`'s dollars | ~55% |
| P8 | `deposit_only` permutation 95% interval **covers 0.50** | ~80% |

Two of v10b's high-confidence predictions missed, so these are pitched lower than they would
have been.

## 0 · Configuration and helpers

In [ ]:
# =====================================================================
# 0 · CONFIGURATION AND HELPERS — v10c
# =====================================================================
# Pure python / numpy / pandas. Spark is imported only in §3 and §11, so
# every analytical function here can be tested without a cluster.
import warnings, time, math
import numpy as np, pandas as pd
from io import BytesIO
from pathlib import Path
from IPython.display import display, HTML, Image
warnings.filterwarnings("ignore")

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
HDFS_V9  = "hdfs://nameservice1/user/pk36814/attrition_v9"
HDFS_V10 = "hdfs://nameservice1/user/pk36814/attrition_v10"
OUT_DIR  = Path(globals().get("OUT_DIR_OVERRIDE",
                              "/projects/DSI/sa15474/repos/pkg/eda/attrition_v10c"))
# TRAP: pathlib collapses hdfs://host/p -> hdfs:/host/p. HDFS paths stay strings.
def v2(n):  return f"{HDFS_V2.rstrip('/')}/{n}"
def v6(n):  return f"{HDFS_V6.rstrip('/')}/{n}"
def v9(n):  return f"{HDFS_V9.rstrip('/')}/{n}"
def v10(n): return f"{HDFS_V10.rstrip('/')}/{n}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED, MAX_ROWS = 20260909, 60
ORIGIN_START_OFF = 18          # OFFSET from M_MIN — m_idx is ABSOLUTE (~24289-24319)
PRIMARY_H = 6
NEG_SAMPLE = 0.15
L2, MIN_TRAIN_POS, CAL_MONTHS = 2.0, 200, 2
LABELS  = ["A_full_exit", "AB_any"]
CONFIGS = ["full", "defend"]

# "money still here": the client holds at least half its normal balance and
# at least $10k. Half mirrors the D_money_move threshold; the floor removes
# relationships too small to be worth a call.
DEFEND_FRAC, DEFEND_MIN = 0.50, 10_000.0
EMPTY_BAL = 1_000.0            # "already empty": under $1k at the alert (absolute)
DRAINED_FRAC = 0.10            # "drained": under 10% of the client's OWN normal balance.
                               # The relative measure is the principled one — an absolute
                               # threshold confuses small clients with drained ones

CAPACITY    = [50, 100, 250, 500, 1000, 2500, 5000]
QUEUE_K     = 1000
P_SAVE_GRID = [0.01, 0.05, 0.10, 0.20, 0.50]
P_SAVE_BASE = 0.10
RM_COST_PER_CALL = 250.0
ANNUALISE   = 12.0/7.0         # overwritten in §3 as 12 / number of test months
ALPHA_GRID  = [0.0, 0.25, 0.5, 0.75, 1.0]
LEAD_GRID   = [1, 2, 3, 4, 6, 9, 12]
PERM_REPS   = 20

V10B_RULE_H = 12               # v10b kept a row whole on ANY of A / D / B within 12 months
V10B_OBSERVED_FACTOR = 0.0506/0.0243   # v10b §3a: realised / predicted mean rate
V10B_TOP = {"deposit_only": 1.7537, "payment_only": 2.3674,
            "both": 1.7332, "all_features": 1.7690}   # v10b weighted top-decile ratios
RES = {}                       # key results, read by the §12 scorecard

HAVE_MPL = True
try:
    import matplotlib
    matplotlib.use("Agg")      # headless is fine — figures are shown as PNG bytes
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    plt.rcParams.update({
        "figure.dpi": 120, "font.size": 9, "axes.spines.top": False,
        "axes.spines.right": False, "axes.grid": True, "grid.color": "#E5E7EB",
        "grid.linewidth": .7, "axes.edgecolor": "#9AA1AC", "axes.labelcolor": "#16181D",
        "xtick.color": "#6B7280", "ytick.color": "#6B7280",
        "figure.facecolor": "white", "axes.facecolor": "white"})
except Exception as e:
    HAVE_MPL = False
    print(f"  matplotlib unavailable ({e}) — charts degrade to tables")
ACC, ACC2, GOOD, WARN, GREY, INK = "#C1440E", "#4A6FA5", "#2F6F4E", "#B8860B", "#9AA1AC", "#16181D"
FS_ORDER = ["deposit_only", "payment_only", "both", "all_features"]
FSCOL = {"deposit_only": GREY, "payment_only": ACC2, "both": GOOD, "all_features": ACC}

def usd(v):
    try: v = float(v)
    except (TypeError, ValueError): return "—"
    if not np.isfinite(v): return "—"
    a, sg = abs(v), ("−" if v < 0 else "")
    if a >= 1e9: return f"{sg}${a/1e9:,.2f}bn"
    if a >= 1e6: return f"{sg}${a/1e6:,.1f}m"
    if a >= 1e3: return f"{sg}${a/1e3:,.0f}k"
    return f"{sg}${a:,.0f}"
def pctf(v, d=1):
    try: return f"{float(v):.{d}%}"
    except (TypeError, ValueError): return "—"
def _dec(s):
    from pyspark.sql import functions as F
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o
def disp(o, title=None, n=None, save=None):
    n = MAX_ROWS if n is None else n
    out = _dec(o).limit(n).toPandas() if hasattr(o, "toPandas") else (
        o.copy() if isinstance(o, pd.DataFrame) else pd.DataFrame(o))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}</div>"))
    display(out.head(n)); return out
def kv(pairs, title=None, save=None):
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"kv(): duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [str(v) for _, v in items]}),
                title=title, n=len(items), save=save)

# ── charts: rendered to PNG bytes, so they display under ANY backend ──
def _show(fig, save=None):
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    png = buf.getvalue()
    if save: (OUT_DIR / f"{save}.png").write_bytes(png)
    display(Image(data=png))
def _fin(fig, title, sub=None, save=None):
    if sub: fig.text(0.005, 0.965, sub, fontsize=8, color="#6B7280", va="top")
    fig.suptitle(title, fontsize=11, fontweight="bold", x=0.005, ha="left", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.93 if sub else 0.96])
    _show(fig, save)
def bar_grouped(df, xlab, series, title, sub=None, ylab="", fmt=usd, save=None,
                colors=None, figsize=(9.5, 4.2), ylim=None):
    """df indexed by the x categories, one column per series."""
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    n = len(series); w = 0.8/n; idx = np.arange(len(df))
    for i, s in enumerate(series):
        vals = df[s].to_numpy(dtype=float)
        b = ax.bar(idx + (i-(n-1)/2)*w, vals, w, label=str(s),
                   color=(colors or {}).get(s))
        for r, v in zip(b, vals):
            if np.isfinite(v):
                ax.text(r.get_x()+r.get_width()/2, v, fmt(v), ha="center",
                        va="bottom", fontsize=6.8)
    ax.set_xticks(idx); ax.set_xticklabels([str(i) for i in df.index], fontsize=8)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    if ylim: ax.set_ylim(*ylim)
    else: ax.margins(y=.18)
    ax.legend(frameon=False, fontsize=8, ncol=min(n, 4))
    _fin(fig, title, sub, save)
def heat(df, title, sub=None, fmt=usd, save=None, xlab="", ylab="", figsize=(9.5, 4.2)):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    V = df.to_numpy(dtype=float)
    im = ax.imshow(V, cmap="OrRd", aspect="auto")
    ax.set_xticks(range(df.shape[1])); ax.set_xticklabels([str(c) for c in df.columns], fontsize=8)
    ax.set_yticks(range(df.shape[0])); ax.set_yticklabels([str(i) for i in df.index], fontsize=8)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(False)
    mx = np.nanmax(V) if np.isfinite(V).any() else 1.0
    for i in range(V.shape[0]):
        for j in range(V.shape[1]):
            if np.isfinite(V[i, j]):
                ax.text(j, i, fmt(V[i, j]), ha="center", va="center", fontsize=7.5,
                        color=("white" if V[i, j] > .62*mx else INK))
    cb = fig.colorbar(im, ax=ax, shrink=.85, pad=.015)
    cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v)))
    cb.ax.yaxis.get_offset_text().set_visible(False)
    _fin(fig, title, sub, save)
def lines(df, title, sub=None, ylab="", xlab="", fmt=None, save=None, colors=None,
          logx=False, figsize=(9.5, 4.2)):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    for c in df.columns:
        ax.plot(df.index, df[c].to_numpy(dtype=float), marker="o", ms=4.5, lw=2,
                color=(colors or {}).get(c), label=str(c))
    if logx:
        ax.set_xscale("log"); ax.set_xticks(list(df.index))
        ax.set_xticklabels([f"{i:,}" for i in df.index], fontsize=8)
    if fmt: ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v)))
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.legend(frameon=False, fontsize=8, ncol=min(len(df.columns), 4))
    _fin(fig, title, sub, save)

# ── model ─────────────────────────────────────────────────────────────
def auc(y, s):
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y) - n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0)/(n1*n0))

def logit_newton(X, y, sw, l2=L2, mi=100, tol=1e-8):
    """Weighted ridge logistic regression by DAMPED Newton.

    The v7-v10b fitter took full Newton (IRLS) steps with no step control and
    can diverge: on the v10c smoke test it drove coefficients past 200,000 and
    saturated every prediction to one value, which scores AUC 0.5000 exactly.
    Here each step is halved until the penalised log-likelihood improves, so
    the objective can only rise. Convergence is judged by the gradient, which
    is the actual optimality condition, not by the step size."""
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64); sw = np.asarray(sw, np.float64)
    k = X.shape[1]; R = l2*np.eye(k); R[0, 0] = 1e-8       # intercept effectively unpenalised
    b = np.zeros(k)
    pbar = float(np.clip(np.average(y, weights=sw), 1e-6, 1-1e-6))
    b[0] = math.log(pbar/(1-pbar))
    def obj(bb):
        eta = X @ bb
        return float(np.sum(sw*(y*eta - np.logaddexp(0.0, eta))) - 0.5*bb @ R @ bb)
    f, it = obj(b), 0
    for it in range(1, mi+1):
        eta = X @ b; mu = 0.5*(1.0 + np.tanh(0.5*eta))
        g = X.T @ (sw*(y-mu)) - R @ b
        Hs = (X.T*(sw*mu*(1-mu))) @ X + R
        try: step = np.linalg.solve(Hs, g)
        except np.linalg.LinAlgError: step = np.linalg.lstsq(Hs, g, rcond=None)[0]
        t = 1.0
        while True:
            bn = b + t*step; fn = obj(bn)
            if fn >= f or t < 1e-10: break
            t *= 0.5
        if fn < f: break                                     # no improving step: at the optimum
        small = np.max(np.abs(bn-b)) < tol
        b, f = bn, fn
        if small: break
    mu = 0.5*(1.0 + np.tanh(0.5*(X @ b)))
    gmax = float(np.max(np.abs(X.T @ (sw*(y-mu)) - R @ b))/len(y))
    return b, dict(converged=gmax < 1e-6, iters=it, max_grad=gmax)

def logit_irls_plain(X, y, l2=L2, mi=25):
    """The v7-v10b fitter, verbatim in logic. Kept ONLY for self-check 1c."""
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
    b = np.zeros(X.shape[1]); R = l2*np.eye(X.shape[1]); R[0, 0] = 0.0
    p0 = float(np.clip(y.mean(), 1e-6, 1-1e-6)); b[0] = math.log(p0/(1-p0))
    for _ in range(mi):
        eta = np.clip(X @ b, -30, 30); mu = 1.0/(1.0+np.exp(-eta))
        v = np.maximum(mu*(1-mu), 1e-6); z = eta + (y-mu)/v; XtW = X.T*v
        b = np.linalg.solve(XtW @ X + R, XtW @ z)
    return b

def fit_spec(tr, cols, l2=L2, wcol="sw"):
    """Inverse-probability-weighted fit. With correct weights the intercept is
    already on the POPULATION scale, so there is NO King & Zeng offset here.
    That offset is only valid when sampling depends on the label alone — v10b's
    design kept some negatives whole, which is exactly what broke it."""
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    sw = (tr[wcol].to_numpy(np.float64) if wcol in tr.columns else np.ones(len(tr)))
    sw = sw/sw.mean()
    keep = X.std(axis=0) > 1e-9
    ck = [c for c, k in zip(cols, keep) if k]
    if not ck or y.sum() < 2: return None
    Xk = X[:, keep]
    mu = np.average(Xk, axis=0, weights=sw)
    sd = np.sqrt(np.average((Xk-mu)**2, axis=0, weights=sw)); sd[sd < 1e-9] = 1.0
    b, info = logit_newton(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y, sw, l2)
    return dict(cols=ck, beta=b[1:]/sd, b0=float(b[0] - np.sum(b[1:]*mu/sd)), **info)

def fit_kz(tr, cols, s=NEG_SAMPLE):
    """The v7-v10b method: unweighted fit + ln(s) intercept offset. Kept only
    so the self-check can show where it breaks."""
    sp = fit_spec(tr.assign(sw=1.0), cols)
    return dict(sp, b0=sp["b0"] + math.log(s)) if sp else None

def predict_p(sp, df):
    if sp is None: return np.full(len(df), np.nan)
    eta = sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
    return 1.0/(1.0+np.exp(-np.clip(eta, -30, 30)))

def pav(x, y, w=None, nbins=200):
    """Weighted isotonic regression (pool-adjacent-violators) on quantile bins.
    Binning keeps it O(nbins); a list-deleting PAV is O(n^2) and does not
    return on a 150k-row slice."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    x, y, w = x[ok], y[ok], w[ok]
    if len(x) == 0: return np.array([0.0]), np.array([0.0])
    nb = int(min(nbins, max(2, len(x)//50)))
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nb+1)))
    if len(edges) < 3:
        return (np.array([float(x.min()), float(x.max())]),
                np.array([float(np.average(y, weights=w))]*2))
    idx = np.clip(np.searchsorted(edges, x, side="right")-1, 0, len(edges)-2)
    g = (pd.DataFrame({"b": idx, "yw": y*w, "xw": x*w, "w": w}).groupby("b")
         .agg(yw=("yw", "sum"), xw=("xw", "sum"), w=("w", "sum")))
    g["x"] = g.xw/g.w; g["y"] = g.yw/g.w
    g = g.sort_values("x")
    sx, sy, sw_ = [], [], []
    for xi, yi, wi in zip(g.x.to_numpy(), g.y.to_numpy(), g.w.to_numpy()):
        sx.append(float(xi)); sy.append(float(yi)); sw_.append(float(wi))
        while len(sy) > 1 and sy[-2] > sy[-1]:
            nw = sw_[-2] + sw_[-1]
            sy[-2] = (sy[-2]*sw_[-2] + sy[-1]*sw_[-1])/nw
            sw_[-2] = nw; sx[-2] = sx[-1]
            sy.pop(); sw_.pop(); sx.pop()
    return np.asarray(sx), np.asarray(sy)

def apply_iso(knots, p):
    kx, ky = knots; p = np.asarray(p, float)
    if len(ky) == 0 or not np.isfinite(ky).any() or np.nanmax(ky) <= 0: return p
    return np.interp(p, kx, ky, left=ky[0], right=ky[-1])

def label(d, defn, H):
    """Forward window only. A row whose panel ends before t+H with no event is
    unobservable and is dropped. AB_any = the earlier of A and B: the money
    leaves whether the account closes or is simply drained."""
    t = pd.to_numeric(d["m_idx"], errors="coerce")
    if defn == "AB_any":
        ev = pd.concat([pd.to_numeric(d["event_A"], errors="coerce"),
                        pd.to_numeric(d["event_B"], errors="coerce")], axis=1).min(axis=1)
    elif defn == "A_full_exit":
        ev = pd.to_numeric(d["event_A"], errors="coerce")
    else:
        raise ValueError(defn)
    y = ((ev > t) & (ev <= t + H)).astype(float)
    keep = ((t + H <= M_MAX) | (y == 1)) & (ev.isna() | (ev > t))
    o = d.loc[keep].copy()
    o["y"] = y.loc[keep].to_numpy(); o["event_m"] = ev.loc[keep].to_numpy()
    return o

# ── the queue ─────────────────────────────────────────────────────────
def first_alerts(df, score_col, K, alpha=0.0, bar_col="bar_med12"):
    """score = p x balance^alpha, ranked within each month; top K alerted.
    Re-flags of the same client are one conversation, credited at the FIRST
    alert. drop_duplicates keeps the whole first row — groupby.first() skips
    NaN per column and mixes months."""
    s = pd.to_numeric(df[score_col], errors="coerce").to_numpy(float)
    if alpha > 0:
        b = np.maximum(pd.to_numeric(df[bar_col], errors="coerce").fillna(0.0).to_numpy(float), 1.0)
        s = s*np.power(b, alpha)
    d = df.assign(_s=s)
    d["_r"] = d.groupby("m_idx")["_s"].rank(ascending=False, method="first", na_option="bottom")
    fl = d[d._r <= K]
    first = fl.sort_values(["cust_pwr_id", "m_idx"]).drop_duplicates("cust_pwr_id", keep="first")
    return fl, first

def savings_grid(frame, tag, alphas=None, caps=None, bar_col="bar_med12", fsets=None):
    """Success applies only to true attriters; on success the balance freezes
    at the first alert, so `bar_now` there is what a save retains."""
    rows = []
    for fs in (fsets or FS_ORDER):
        for alpha in (ALPHA_GRID if alphas is None else alphas):
            for K in (CAPACITY if caps is None else caps):
                _, first = first_alerts(frame, f"p_{fs}", K, alpha, bar_col)
                tp = first[first.y == 1]
                reach = float(tp.bar_now.sum()); conv = int(len(first))
                for ps in P_SAVE_GRID:
                    saved = reach*ps; cost = conv*RM_COST_PER_CALL
                    rows.append(dict(cfg=tag, feature_set=fs, alpha=alpha, k=K, p_save=ps,
                                     rank_on=bar_col, conversations=conv,
                                     tp_clients=int(len(tp)),
                                     precision=float(len(tp)/max(conv, 1)),
                                     dollars_reached=reach,
                                     saved_annualised=saved*ANNUALISE,
                                     rm_cost_annualised=cost*ANNUALISE,
                                     net_annualised=(saved-cost)*ANNUALISE,
                                     breakeven_p_save=cost/max(reach, 1.0)))
    return pd.DataFrame(rows)

def frontier(d):
    """Efficient frontier of (conversations, dollars), anchored at the origin.
    Step 1 keeps a point only if it ADDS money — a costlier point with no more
    money is dominated. (v10b had this inverted and kept only the last point.)
    Step 2 takes the upper concave envelope; its slopes are the marginal
    dollars per extra conversation, decreasing by construction."""
    d = d.sort_values(["conversations", "dollars_reached"], ascending=[True, False])
    pts, best = [(0.0, 0.0, 0)], 0.0
    for r in d.itertuples():
        if float(r.dollars_reached) > best and float(r.conversations) > 0:
            pts.append((float(r.conversations), float(r.dollars_reached), int(r.k)))
            best = float(r.dollars_reached)
    hull = []
    for pt in pts:
        while len(hull) >= 2:
            (x0, y0, _), (x1, y1, _) = hull[-2], hull[-1]
            if (y1-y0)*(pt[0]-x1) <= (pt[1]-y1)*(x1-x0): hull.pop()
            else: break
        hull.append(pt)
    out = pd.DataFrame(hull, columns=["conversations", "dollars_reached", "k"])
    out["marginal_per_conversation"] = out.dollars_reached.diff()/out.conversations.diff()
    return out
print(f"helpers ready · charts {'on' if HAVE_MPL else 'OFF'} · out {OUT_DIR}")


## 1 · Self-checks — against the design, not just the arithmetic

v10b's calibration self-check sampled on the label alone, passed, and the real pipeline still came
out 2× low — because the real pipeline also kept rows whole for *other* events. A check that does
not reproduce the design cannot catch a design defect. These do:

- **(a)** A synthetic population with a second event that shares a driver with the label — the
  analogue of B clients who drain without closing. Rows are kept whole on *either* event, exactly
  as v10b did. The v7–v10b method (King & Zeng offset) should come out low by roughly the
  effective-inclusion ratio and bias the shared-driver coefficient; the IPW fit should recover
  both the level and the coefficients.
- **(b)** The frontier on a known concave curve with one dominated point, which must be dropped
  while every other point survives.
- **(c)** The fitter, on a case where the v7–v10b IRLS diverges. Found while testing this
  notebook: on a synthetic panel the old fitter drove coefficients past 200,000 and saturated
  every prediction, which scores an AUC of exactly 0.5000.

Check (a) also renders a chart. **If no image appears below it, stop** — the chart fix has not
taken.

In [ ]:
# =====================================================================
# 1 · SELF-CHECKS                                        [OUTPUT BLOCK 1]
# =====================================================================
# (a) IPW against King & Zeng under v10b's actual kind of design
rng = np.random.default_rng(7)
n = 400_000
Xs = rng.normal(size=(n, 3))
y_ = (rng.random(n) < 1/(1+np.exp(-(-3.4 + 0.8*Xs[:, 0] - 0.5*Xs[:, 1])))).astype(float)
s_ = (rng.random(n) < 1/(1+np.exp(-(-2.2 + 0.9*Xs[:, 0] + 0.7*Xs[:, 2])))).astype(float)
whole = (y_ == 1) | (s_ == 1)                     # kept whole on EITHER event
keep_ = whole | (rng.random(n) < NEG_SAMPLE)
POP = pd.DataFrame(Xs, columns=["x0", "x1", "x2"]).assign(y=y_, sw=1.0)
SMP = POP[keep_].copy()
SMP["sw"] = np.where(whole[keep_], 1.0, 1.0/NEG_SAMPLE)
_c = ["x0", "x1", "x2"]
_truth, _kz, _ipw = fit_spec(POP, _c), fit_kz(SMP, _c), fit_spec(SMP, _c)
_neg = POP.y == 0
_f = float(whole[_neg.to_numpy()].mean())
_eff = _f + NEG_SAMPLE*(1-_f)
CHK = pd.DataFrame([
    dict(method="truth (full population)", mean_pred=predict_p(_truth, POP).mean(),
         **{f"coef_{c}": b for c, b in zip(_truth["cols"], _truth["beta"])}),
    dict(method="King & Zeng offset (v7-v10b)", mean_pred=predict_p(_kz, POP).mean(),
         **{f"coef_{c}": b for c, b in zip(_kz["cols"], _kz["beta"])}),
    dict(method="IPW (v10c)", mean_pred=predict_p(_ipw, POP).mean(),
         **{f"coef_{c}": b for c, b in zip(_ipw["cols"], _ipw["beta"])})])
CHK["ratio_true_over_pred"] = y_.mean()/CHK.mean_pred
disp(CHK.round(4), title=f"1a &middot; <b>Self-check against the design.</b> True rate "
     f"{y_.mean():.3%}; {_f:.1%} of negatives kept whole, so the effective negative inclusion is "
     f"{_eff:.3f}, not {NEG_SAMPLE}. King &amp; Zeng should be low by about "
     f"{_eff/NEG_SAMPLE:.2f}&times; and bias <code>coef_x0</code>, the shared driver",
     save="v10c_selfcheck_ipw")
assert abs(CHK.ratio_true_over_pred.iloc[2] - 1) < 0.08, "IPW failed to recover the population rate"
assert abs(CHK.ratio_true_over_pred.iloc[1] - 1) > abs(CHK.ratio_true_over_pred.iloc[2] - 1), \
    "the self-check did not reproduce the v10b defect — it would not have caught it"
assert abs(CHK.coef_x0.iloc[2] - CHK.coef_x0.iloc[0]) < abs(CHK.coef_x0.iloc[1] - CHK.coef_x0.iloc[0]), \
    "IPW should recover the shared-driver coefficient better than the offset method"

if HAVE_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.4))
    axes[0].bar(CHK.method.str.split(" ").str[0], CHK.mean_pred, color=[INK, GREY, ACC])
    axes[0].axhline(y_.mean(), ls=":", color=INK)
    axes[0].set_title("mean predicted rate (dotted = truth)", fontsize=9)
    axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.1%}"))
    axes[1].bar(CHK.method.str.split(" ").str[0], CHK.coef_x0, color=[INK, GREY, ACC])
    axes[1].axhline(CHK.coef_x0.iloc[0], ls=":", color=INK)
    axes[1].set_title("coefficient on the shared driver", fontsize=9)
    _fin(fig, "1a · The v10b defect, reproduced and fixed",
         "if you can see this chart, the rendering fix works", save="v10c_selfcheck_ipw")

# (b) frontier on a known curve with one dominated point
_pts = pd.DataFrame(dict(k=[50, 100, 250, 500, 1000, 2500, 5000],
                         conversations=[100, 180, 400, 800, 1500, 3000, 6000],
                         dollars_reached=[10e6, 16e6, 28e6, 40e6, 50e6, 49e6, 55e6]))
_fr = frontier(_pts)
disp(_fr.assign(dollars_reached=_fr.dollars_reached.map(usd),
                marginal_per_conversation=_fr.marginal_per_conversation.map(usd)),
     title="1b &middot; Frontier self-check. K=2,500 costs more than K=1,000 and holds less, "
           "so it must be dropped; every other point survives", save="v10c_selfcheck_frontier")
assert 2500 not in _fr.k.tolist(), "dominated point survived"
assert len(_fr) == 7, f"expected origin + 6 points, got {len(_fr)} — the filter is wrong again"
assert _fr.marginal_per_conversation.dropna().is_monotonic_decreasing

# (c) the fitter: a case on which the v7-v10b IRLS diverges
_r = np.random.default_rng(11); _n = 60_000
_frac = np.where(_r.random(_n) < .03, .02, 1.0)*_r.uniform(.85, 1.15, _n)
_x = np.log(_frac + .01) + _r.normal(0, .3, _n)            # a "balance already gone" feature
_y = (_r.random(_n) < np.where(_frac < .5, .35, .01)).astype(float)
_X = np.column_stack([np.ones(_n), (_x - _x.mean())/_x.std()])
_bp = logit_irls_plain(_X, _y)
_bd, _info = logit_newton(_X, _y, np.ones(_n))
kv([("v7-v10b IRLS · largest |coefficient| after 25 steps", f"{np.abs(_bp).max():,.1f}"),
    ("damped Newton · largest |coefficient|", f"{np.abs(_bd).max():.3f}"),
    ("damped Newton · converged / iterations", f"{_info['converged']} / {_info['iters']}"),
    ("damped Newton · max gradient per observation", f"{_info['max_grad']:.1e}")],
   title="1c &middot; <b>The fitter.</b> Full Newton steps without step control diverge on a "
         "steep feature like &lsquo;balance already gone&rsquo;; damped steps cannot",
   save="v10c_selfcheck_fitter")
assert _info["converged"] and np.abs(_bd).max() < 50, "damped Newton failed its self-check"
print("  self-checks passed")


## 2 · What α is

Every month the queue ranks roughly 78,000 at-risk clients and hands the top **K** to relationship
managers. The score is

$$\text{score} = p \times \text{balance}^{\,\alpha}$$

- **p** — the calibrated probability that the client leaves within the horizon
- **balance** — `bar_med12`, the client's 12-month median balance: its normal relationship size
- **α** — how much size matters, set between 0 and 1

**α = 0** gives `score = p`: *call whoever is most likely to leave.* A $5k client and a $50m client
count the same. This maximises the **number** of departing clients caught — it is the precision
queue every version before v9 optimised.

**α = 1** gives `score = p × balance`, which is **expected dollars at risk**: *call where the most
money is likely to leave.* A $50m client with a 1% chance ($500k expected) outranks a $100k client
with a 40% chance ($40k expected).

**In between**, size matters with diminishing returns. A client ten times larger gets 10^α times
the weight:

| α | 0 | 0.25 | 0.5 | 0.75 | 1 |
|---|---|---|---|---|---|
| weight of a 10× larger client | 1× | 1.8× | 3.2× | 5.6× | 10× |

### Worked example

Client **A**: 30% likely to leave, $50k. Client **B**: 3% likely to leave, $2m.

| α | score A | score B | who gets the call |
|---|---|---|---|
| 0 | 0.30 | 0.03 | A |
| 0.5 | 67 | 42 | A |
| 0.75 | 1,003 | 1,595 | **B** |
| 1 | $15k expected | $60k expected | **B** |

B overtakes A at α = ln 10 / ln 40 ≈ **0.62**.

### Why not always α = 1?

It is the principled reference: with a calibrated `p`, and a balance equal to what a save would
actually retain, `p × balance` *is* expected dollars. It falls short in practice for three reasons,
each of which has shown up in this programme:

1. **We rank on what the client normally holds and credit what it still holds.** The score uses
   `bar_med12`; a save retains `bar_now`. For a client already draining, the first overstates the
   second — the empty-account problem in a different form. §8 tests ranking on `bar_now` directly.
2. **Calibration errors multiply.** If `p` is 2× low in the top decile, expected dollars are 2× low,
   and not uniformly across client sizes. That is why §4–5 fix calibration first.
3. **Heavy tails.** The top 1% of departing clients hold 64.5% of at-risk dollars. At α = 1 a few
   large clients with modest `p` can take many slots, and one hit or miss swings the total.

So α is a **tuning knob chosen on realised dollars in the test folds**, with α = 1 as the
reference. In v10b the best value ranged from 0.25 to 1.0 depending on the feature set.

In [ ]:
# =====================================================================
# 2 · alpha, illustrated                                 [OUTPUT BLOCK 2]
# =====================================================================
_pA, _bA, _pB, _bB = 0.30, 50_000.0, 0.03, 2_000_000.0
_cross = math.log(_pA/_pB)/math.log(_bB/_bA)
if HAVE_MPL:
    al = np.linspace(0, 1, 201)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    ax = axes[0]
    ax.plot(al, _pA*_bA**al, lw=2.2, color=ACC2, label="A · 30% likely · $50k")
    ax.plot(al, _pB*_bB**al, lw=2.2, color=ACC, label="B · 3% likely · $2m")
    ax.set_yscale("log")
    ax.axvline(_cross, color=INK, ls=":", lw=1.2)
    ax.text(_cross, ax.get_ylim()[0]*2.5, f"  B overtakes A at α ≈ {_cross:.2f}", fontsize=8)
    ax.set_xlabel("α"); ax.set_ylabel("score  p × balance^α")
    ax.legend(frameon=False, fontsize=8)
    ax = axes[1]
    _al = [0, 0.25, 0.5, 0.75, 1.0]
    ax.bar([str(a) for a in _al], [10**a for a in _al], color=ACC2)
    for i, a in enumerate(_al):
        ax.text(i, 10**a, f"{10**a:.1f}×", ha="center", va="bottom", fontsize=8)
    ax.set_xlabel("α"); ax.set_ylabel("weight of a client 10× larger")
    ax.margins(y=.15)
    _fin(fig, "2 · α decides how much size matters in the queue",
         "α = 0 counts clients; α = 1 counts expected dollars", save="v10c_alpha_explained")
kv([("score", "p × balance^α — ranked within each month, top K called"),
    ("alpha = 0", "rank on probability only — maximises departing clients caught"),
    ("alpha = 1", "rank on expected dollars at risk — maximises money in front of RMs"),
    ("crossover in the worked example", f"α ≈ {_cross:.2f}"),
    ("how it is chosen", "empirically, on realised dollars in the test folds"),
    ("balance used", "bar_med12 by default; §8 also tests bar_now")],
   title="2 &middot; α in one table", save="v10c_alpha_table")


## 3 · The sampling design, made explicit

The training frame is case-control. Every row that is a positive for a label this notebook fits is
kept; every other row is kept with probability 0.15. v10b's rule also kept any row with a **D or B
event within 12 months**, which made a large share of *negatives* certain inclusions. The King &
Zeng offset assumes every negative had a 15% chance. For those rows it was 100%, so the offset
over-corrected and every probability came out low — and those rows, mostly B clients who drain
without closing, also sat among the A-negatives at about six times their population weight.

**v10c.** Each row carries `sw` = 1 / its inclusion probability, computed from **the same
expression that did the filtering**, so the two cannot drift apart. The fit is weighted logistic
regression with no intercept offset, and the isotonic map uses the same weights. That is correct
under any sampling rule, v10b's included.

The audit at the end of this block **measures** v10b's rule on this data: the share of population
negatives it kept whole, and the miscalibration factor that implies. v10b observed 2.08. If the
audit lands near that, the diagnosis is confirmed rather than argued (P1).

In [ ]:
# =====================================================================
# 3 · DATA                                               [OUTPUT BLOCK 3]
# =====================================================================
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
if "spark" not in dir():
    spark = (SparkSession.builder.appName("pkg_attrition_eda_v10c")
             .config("spark.sql.shuffle.partitions", "800")
             .config("spark.sql.execution.arrow.pyspark.enabled", "false")
             # KEEP OFF: PySpark 3.3's arrow path uses np.object0 / np.bool8,
             # both removed in numpy 2.0
             .enableHiveSupport().getOrCreate())
def collect_pd(sdf, lbl=""):
    t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {lbl}: {len(out):,} x {out.shape[1]} in {time.time()-t0:,.0f}s")
    return out

t_data = time.time()
cust_month = spark.read.parquet(v2("panel_customer_month"))
M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]
ORIGINS = list(range(M_MIN + ORIGIN_START_OFF, M_MAX - PRIMARY_H + 1))
assert 1 <= len(ORIGINS) <= 24, (f"{len(ORIGINS)} origins — m_idx is ABSOLUTE "
                                 f"({M_MIN}-{M_MAX}); ORIGIN_START_OFF is an OFFSET")
ANNUALISE = 12.0/len(ORIGINS)
print(f"  m_idx {M_MIN}-{M_MAX} · origins {ORIGINS[0]}-{ORIGINS[-1]} ({len(ORIGINS)} test months)")

lab  = spark.read.parquet(v6("labels_customer")).persist(StorageLevel.DISK_ONLY)
BAL  = spark.read.parquet(v9("balance")).persist(StorageLevel.DISK_ONLY)
RISK = spark.read.parquet(v9("risk_set_v9"))
ALLC = RISK.columns
_missing = {"cust_pwr_id", "m_idx", "event_A", "event_B", "bar_now", "bar_med12"} - set(ALLC)
assert not _missing, f"risk_set_v9 lacks {_missing}"

# feature sets — identical construction to v10 / v10b
NEWP = ("cptyn_", "cptya_", "fin_out2", "fin_in", "tim_", "railmix", "conc_",
        "selfpay", "acc_", "bl_")
def _pair(ld): return sorted(set(ld + [f"md_{c[3:]}" for c in ld if f"md_{c[3:]}" in ALLC]))
def _blk(*pfx): return _pair(sorted([c for c in ALLC if any(c.startswith("ld_"+p) for p in pfx)]))
V7_LD = sorted([c for c in ALLC if c.startswith("ld_")
                and not any(c.startswith("ld_"+p) for p in NEWP)])
DEPOSIT = _pair([c for c in V7_LD if c == "ld_bal_live"]) + _blk("bl_") + _blk("acc_")
PAYMENT = _pair([c for c in V7_LD if c != "ld_bal_live"])
BOTH    = sorted(set(DEPOSIT + PAYMENT))
ALLF    = sorted(set(BOTH + _blk("fin_in") + _blk("cptya_out") + _blk("fin_out2") +
                     [c for c in ALLC if "rec_" in c and c.startswith(("ld_", "md_"))]))
FSETS = {"deposit_only": DEPOSIT, "payment_only": PAYMENT, "both": BOTH, "all_features": ALLF}
assert not (set(DEPOSIT) & set(PAYMENT)), "deposit and payment sets must be disjoint"

DL = spark.read.parquet(v10("labels_D")).withColumnRenamed("q_D_money_move", "event_D")
NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B", "bar_now", "bar_med12"] + ALLF)
              & set(ALLC))
RS = RISK.select(*NEED).join(DL, "cust_pwr_id", "left")

def _ahead(c, h):
    return F.coalesce(F.col(c).between(F.col("m_idx") + 1, F.col("m_idx") + h), F.lit(False))
# THE RULE. Kept whole = a positive for ANY label fitted here (A, or B for
# AB_any) at the primary horizon. Everything else kept with prob NEG_SAMPLE.
# `sw` is derived from the SAME expression that filters, so it cannot drift.
KEEP_WHOLE = _ahead("event_A", PRIMARY_H) | _ahead("event_B", PRIMARY_H)
# v10b's rule, carried only so the audit can measure what it did
V10B_WHOLE = (_ahead("event_A", V10B_RULE_H) | _ahead("event_D", V10B_RULE_H) |
              _ahead("event_B", V10B_RULE_H))
TRS = (RS.filter(F.col("m_idx") <= max(ORIGINS) - PRIMARY_H)
       .withColumn("kept_whole", KEEP_WHOLE.cast("int"))
       .withColumn("v10b_whole", V10B_WHOLE.cast("int"))
       .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                   F.col("m_idx").cast("string"), F.lit(SEED)))) % 100000)/100000.0)
       .filter((F.col("kept_whole") == 1) | (F.col("_u") < NEG_SAMPLE))
       .withColumn("sw", F.when(F.col("kept_whole") == 1, F.lit(1.0))
                          .otherwise(F.lit(1.0/NEG_SAMPLE)))
       .drop("_u"))
TR_RAW = collect_pd(TRS, "TRAIN")
TE_RAW = {t: collect_pd(RS.filter(F.col("m_idx") == t), f"TEST {t}") for t in ORIGINS}

def prep(d):
    d = d.copy()
    for c in d.columns:
        if c.startswith("ld_"):   d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
        elif c.startswith("md_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
    for c in ["bar_now", "bar_med12"]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0).clip(lower=0)
    for c, dv in [("sw", 1.0), ("kept_whole", 1), ("v10b_whole", 0)]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(dv) if c in d.columns else dv
    d["defend"] = (d.bar_now >= DEFEND_FRAC*d.bar_med12) & (d.bar_now >= DEFEND_MIN)
    return d
TRw = prep(TR_RAW); TE = {k: prep(v) for k, v in TE_RAW.items()}
del TR_RAW, TE_RAW
print(f"  data ready in {time.time()-t_data:,.0f}s")


In [ ]:
# =====================================================================
# 3b · SAMPLING AUDIT + RISK-SET SHAPE                   [OUTPUT BLOCK 4]
# =====================================================================
_a = label(TRw, "A_full_exit", PRIMARY_H)
_n = _a[_a.y == 0]
_wn = float(_n.sw.sum())
f_v10c = float(_n.loc[_n.kept_whole == 1, "sw"].sum()/_wn)     # weighted = population share
f_v10b = float(_n.loc[_n.v10b_whole == 1, "sw"].sum()/_wn)
eff_v10b = f_v10b + NEG_SAMPLE*(1 - f_v10b)
RES["audit_factor"] = eff_v10b/NEG_SAMPLE

_te = pd.concat([label(TE[t], "A_full_exit", PRIMARY_H) for t in ORIGINS], ignore_index=True)
_pos = _te[_te.y == 1]
kv([("training rows collected", f"{len(TRw):,}"),
    ("A positives in training (H=6)", f"{int(_a.y.sum()):,}"),
    ("population negatives kept whole — v10c rule", pctf(f_v10c)),
    ("population negatives kept whole — v10b rule", pctf(f_v10b)),
    ("v10b effective negative inclusion (assumed 0.15)", f"{eff_v10b:.3f}"),
    ("IMPLIED v10b miscalibration factor", f"{RES['audit_factor']:.2f}x"),
    ("OBSERVED v10b miscalibration factor", f"{V10B_OBSERVED_FACTOR:.2f}x"),
    ("test client-months (all folds)", f"{len(_te):,}"),
    ("... of which money still here", pctf(_te.defend.mean())),
    ("test attriters (client-months, y=1)", f"{len(_pos):,}"),
    ("... already empty at t (bar_now < $1k)", pctf((_pos.bar_now < EMPTY_BAL).mean())),
    ("... money still here at t", pctf(_pos.defend.mean())),
    ("share of attriter balance sitting in money-still-here rows",
     pctf(_pos.loc[_pos.defend, "bar_now"].sum()/max(_pos.bar_now.sum(), 1)))],
   title="3b &middot; <b>The sampling audit.</b> If the implied factor lands near the observed "
         "2.08&times;, the v10b calibration defect is explained by the design, measured rather "
         "than argued. The lower rows show how much of the test set is attriters whose money "
         "has <i>already</i> gone", save="v10c_audit")


In [ ]:
# =====================================================================
# 4 · FIT — IPW + weighted isotonic, two risk sets        [OUTPUT BLOCK 5]
# =====================================================================
# full   : every at-risk client-month, as in v10 / v10b
# defend : only client-months where the money is still here, in training AND
#          test. The model can no longer earn credit for predicting the
#          closure of an account that is already empty.
# The isotonic map is fitted on the last CAL_MONTHS of each TRAINING window —
# still inside t <= T-H, so nothing from the test month is seen.
def run(defn, cfg):
    folds, scored = [], []
    for T in ORIGINS:
        H = PRIMARY_H
        if T + H > M_MAX: continue
        trf = label(TRw[TRw.m_idx <= T - H], defn, H)
        if cfg == "defend": trf = trf[trf.defend]
        if len(trf) == 0 or trf.y.sum() < MIN_TRAIN_POS: continue
        cut = trf.m_idx.max() - CAL_MONTHS
        fit_df, cal_df = trf[trf.m_idx <= cut], trf[trf.m_idx > cut]
        if len(cal_df) < 3000 or cal_df.y.sum() < 20 or fit_df.y.sum() < MIN_TRAIN_POS:
            fit_df = cal_df = trf                   # degrade rather than fail
        te = label(TE[T], defn, H)
        if cfg == "defend": te = te[te.defend]
        if len(te) == 0 or te.y.sum() < 1: continue
        out = te[["cust_pwr_id", "m_idx", "y", "event_m", "bar_now", "bar_med12",
                  "defend"]].copy()
        out["defn"], out["cfg"] = defn, cfg
        yv, dm = te.y.to_numpy(), te.defend.to_numpy()
        for fs, cols in FSETS.items():
            cols = [c for c in cols if c in fit_df.columns]
            sp = fit_spec(fit_df, cols)
            praw = predict_p(sp, te)
            knots = pav(predict_p(sp, cal_df), cal_df.y.to_numpy(), cal_df.sw.to_numpy())
            out[f"praw_{fs}"] = praw
            out[f"p_{fs}"] = apply_iso(knots, praw)
            folds.append(dict(defn=defn, cfg=cfg, origin=T, feature_set=fs,
                              n_feat=len(sp["cols"]) if sp else 0, n_train=len(fit_df),
                              train_pos=int(fit_df.y.sum()), test_rows=len(te),
                              test_pos=int(yv.sum()), auc=auc(yv, praw),
                              converged=bool(sp["converged"]) if sp else False,
                              iters=sp["iters"] if sp else 0,
                              max_grad=sp["max_grad"] if sp else np.nan,
                              auc_on_defendable=(auc(yv[dm], praw[dm])
                                                 if cfg == "full" else np.nan)))
        scored.append(out)
    return (pd.DataFrame(folds),
            pd.concat(scored, ignore_index=True) if scored else pd.DataFrame())

t0 = time.time(); FD, SC = {}, {}
for defn in LABELS:
    for cfg in CONFIGS:
        FD[(defn, cfg)], SC[(defn, cfg)] = run(defn, cfg)
        print(f"  {defn:12s} {cfg:7s} {len(SC[(defn, cfg)]):>9,} scored rows "
              f"({time.time()-t0:,.0f}s)")
FOLDS = pd.concat(FD.values(), ignore_index=True)
FOLDS.to_csv(OUT_DIR / "v10c_folds.csv", index=False)
kv([("fits", f"{len(FOLDS):,}"),
    ("converged (max gradient per observation < 1e-6)",
     f"{int(FOLDS.converged.sum()):,} of {len(FOLDS):,}"),
    ("iterations, median / max", f"{FOLDS.iters.median():.0f} / {FOLDS.iters.max():.0f}"),
    ("largest final gradient", f"{FOLDS.max_grad.max():.1e}")],
   title="4 &middot; <b>Every fit, checked for convergence</b>", save="v10c_convergence")
_bad = FOLDS[~FOLDS.converged]
assert _bad.empty, ("non-converged fits — do not read anything downstream:\n"
                    + _bad[["defn", "cfg", "origin", "feature_set", "iters", "max_grad"]].to_string())
for (d, c), s in SC.items():       # persist, so the next version never refits
    try: s.to_parquet(OUT_DIR / f"v10c_scored_{d}_{c}.parquet", index=False)
    except Exception as e: print(f"  (scored frame not persisted: {type(e).__name__})"); break


In [ ]:
# =====================================================================
# 5 · CALIBRATION                                        [OUTPUT BLOCK 6]
# =====================================================================
S = SC[("A_full_exit", "full")]
rows = []
for fs in FS_ORDER:
    for tag, pre in [("IPW raw", "praw_"), ("IPW + isotonic", "p_")]:
        d = S[["y", pre+fs]].dropna().rename(columns={pre+fs: "p"})
        d["dec"] = pd.qcut(d.p.rank(method="first"), 10, labels=False) + 1
        g = d.groupby("dec", as_index=False).agg(pred=("p", "mean"), real=("y", "mean"))
        top = g[g.dec == 10].iloc[0]
        rows.append(dict(feature_set=fs, version=tag, mean_pred=float(d.p.mean()),
                         mean_real=float(d.y.mean()),
                         level_ratio=float(d.y.mean()/max(d.p.mean(), 1e-12)),
                         calib_error=float(np.abs(g.real-g.pred).sum()/g.real.sum()),
                         top_pred=float(top.pred), top_real=float(top.real),
                         top_ratio=float(top.real/max(top.pred, 1e-12))))
CAL = pd.DataFrame(rows)
_iso = CAL[CAL.version == "IPW + isotonic"]
RES["cal_top_range"] = (float(_iso.top_ratio.min()), float(_iso.top_ratio.max()))
disp(CAL.round(4), title="5a &middot; <b>Calibration after the IPW fix.</b> "
     "<code>top_ratio</code> near 1.00 is the target — that decile is the queue. v10b's "
     "weighted isotonic sat at 1.73&ndash;2.37", n=10, save="v10c_calibration")
_cmp = pd.DataFrame({"v10b weighted isotonic": pd.Series(V10B_TOP),
                     "v10c IPW raw": CAL[CAL.version == "IPW raw"].set_index("feature_set").top_ratio,
                     "v10c IPW + isotonic": _iso.set_index("feature_set").top_ratio}).reindex(FS_ORDER)
bar_grouped(_cmp, "feature set", list(_cmp.columns),
            "5a · Top-decile calibration ratio — 1.00 is correct",
            "realised ÷ predicted rate in the decile the queue works from",
            ylab="ratio", fmt=lambda v: f"{v:.2f}", save="v10c_calib_ratio",
            colors={"v10b weighted isotonic": GREY, "v10c IPW raw": ACC2,
                    "v10c IPW + isotonic": ACC})
if HAVE_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
    for ax, (tag, pre) in zip(axes, [("IPW raw", "praw_"), ("IPW + isotonic", "p_")]):
        for fs in FS_ORDER:
            d = S[["y", pre+fs]].dropna().rename(columns={pre+fs: "p"})
            d["dec"] = pd.qcut(d.p.rank(method="first"), 10, labels=False) + 1
            g = d.groupby("dec").agg(pred=("p", "mean"), real=("y", "mean"))
            ax.plot(g.pred, g.real, marker="o", ms=4, lw=1.6, color=FSCOL[fs], label=fs)
        lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([0, lim], [0, lim], ls=":", color=INK, lw=1)
        ax.set_title(tag, fontsize=9.5); ax.set_xlabel("predicted")
    axes[0].set_ylabel("realised"); axes[0].legend(frameon=False, fontsize=7.5)
    _fin(fig, "5b · Reliability by decile — points on the diagonal are calibrated",
         "A_full_exit · full risk set · every fold", save="v10c_reliability")


## 6 · When does each queue arrive?

The direct test of the timing story. For each feature set, take the departing clients the queue
actually reaches and look at their balance **at the moment of the first alert**:

- `share_drained` — balance under 10% of the client's own normal balance when the RM would have
  called. **This is the primary measure**: it is relative, so a small client is not mistaken for a
  drained one
- `share_already_empty` — balance under $1k, the absolute version
- `median_share_still_on_book` — current balance as a share of the client's normal balance
- `dollars_per_client` — what a save would retain, on average

If deposit features are lagging indicators, `deposit_only` at α = 0 should reach mostly drained
accounts (P3) and `payment_only` mostly live ones (P4).

In [ ]:
# =====================================================================
# 6 · ARRIVAL TIMING                                     [OUTPUT BLOCK 7]
# =====================================================================
S = SC[("A_full_exit", "full")]
rows, SHARES = [], {}
for fs in FS_ORDER:
    for alpha in [0.0, 1.0]:
        _, first = first_alerts(S, f"p_{fs}", QUEUE_K, alpha)
        tp = first[first.y == 1]
        sh = (tp.bar_now/tp.bar_med12.where(tp.bar_med12 > 0)).clip(upper=2.0)
        if alpha == 0.0: SHARES[fs] = sh.dropna().to_numpy()
        rows.append(dict(feature_set=fs, alpha=alpha, conversations=len(first),
                         departing_clients=len(tp),
                         precision=len(tp)/max(len(first), 1),
                         median_lead_months=(float((tp.event_m - tp.m_idx).median())
                                             if len(tp) else np.nan),
                         median_share_still_on_book=float(sh.median()) if len(sh) else np.nan,
                         share_drained=(float((tp.bar_now < DRAINED_FRAC*tp.bar_med12).mean())
                                        if len(tp) else np.nan),
                         share_already_empty=(float((tp.bar_now < EMPTY_BAL).mean())
                                              if len(tp) else np.nan),
                         dollars_reached=float(tp.bar_now.sum()),
                         dollars_per_client=float(tp.bar_now.sum()/max(len(tp), 1))))
TIMING = pd.DataFrame(rows)
TIMING.to_csv(OUT_DIR / "v10c_timing.csv", index=False)
_t0 = TIMING[TIMING.alpha == 0.0].set_index("feature_set")
RES["drained_dep_a0"] = float(_t0.loc["deposit_only", "share_drained"])
RES["drained_pay_a0"] = float(_t0.loc["payment_only", "share_drained"])
disp(TIMING.assign(precision=TIMING.precision.map(pctf),
                   share_drained=TIMING.share_drained.map(pctf),
                   median_share_still_on_book=TIMING.median_share_still_on_book.map(pctf),
                   share_already_empty=TIMING.share_already_empty.map(pctf),
                   dollars_reached=TIMING.dollars_reached.map(usd),
                   dollars_per_client=TIMING.dollars_per_client.map(usd)),
     title=f"6a &middot; <b>When does each queue arrive?</b> Departing clients reached at "
           f"{QUEUE_K:,} alerts a month, with their balance at the first alert", n=10,
     save="v10c_arrival")
bar_grouped(_t0[["share_drained"]].rename(columns={"share_drained": "α = 0"})
            .join(TIMING[TIMING.alpha == 1.0].set_index("feature_set")[["share_drained"]]
                  .rename(columns={"share_drained": "α = 1"})).reindex(FS_ORDER),
            "feature set", ["α = 0", "α = 1"],
            "6a · Share of reached attriters already drained when the RM calls",
            f"balance under {DRAINED_FRAC:.0%} of normal at the first alert · "
            f"{QUEUE_K:,} alerts/month", ylab="share drained", fmt=lambda v: f"{v:.0%}",
            save="v10c_drained_share",
            colors={"α = 0": ACC2, "α = 1": ACC}, ylim=(0, 1.08))
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    bins = np.linspace(0, 2, 41)
    for fs in FS_ORDER:
        if len(SHARES.get(fs, [])):
            ax.hist(SHARES[fs], bins=bins, histtype="step", lw=2, density=True,
                    color=FSCOL[fs], label=fs)
    for xv, lab_ in [(DRAINED_FRAC, "  drained"), (DEFEND_FRAC, "  half")]:
        ax.axvline(xv, color=INK, ls=":", lw=1)
        ax.text(xv, ax.get_ylim()[1]*.92, lab_, fontsize=8)
    ax.set_xlabel("balance at the first alert ÷ normal balance (1.0 = untouched)")
    ax.set_ylabel("density"); ax.legend(frameon=False, fontsize=8)
    _fin(fig, "6b · How much of the relationship is left when each queue calls",
         "α = 0 · departing clients only · mass near 0 means the money has already gone",
         save="v10c_share_hist")


In [ ]:
# =====================================================================
# 7 · AUC WITH AND WITHOUT THE EMPTY ACCOUNTS             [OUTPUT BLOCK 8]
# =====================================================================
_F = FOLDS[FOLDS.defn == "A_full_exit"]
AUCD = (_F[_F.cfg == "full"].groupby("feature_set")
        .agg(auc_full_book=("auc", "mean"),
             full_model_where_money_still_here=("auc_on_defendable", "mean"))
        .join(_F[_F.cfg == "defend"].groupby("feature_set")
              .agg(retrained_where_money_still_here=("auc", "mean")))
        .reindex(FS_ORDER))
AUCD["drop_when_empties_removed"] = AUCD.auc_full_book - AUCD.full_model_where_money_still_here
RES["auc_drop_dep"] = float(AUCD.loc["deposit_only", "drop_when_empties_removed"])
RES["auc_drop_pay"] = float(AUCD.loc["payment_only", "drop_when_empties_removed"])
disp(AUCD.round(4).reset_index(),
     title="7a &middot; <b>How much of each model's AUC came from predicting empty accounts?</b> "
           "Column 2 scores the same model only on client-months where the money is still here; "
           "column 3 retrains on them. A large drop means the AUC was earned on accounts nobody "
           "could save", save="v10c_auc_decomposition")
bar_grouped(AUCD[["auc_full_book", "full_model_where_money_still_here",
                  "retrained_where_money_still_here"]],
            "feature set", ["auc_full_book", "full_model_where_money_still_here",
                            "retrained_where_money_still_here"],
            "7a · AUC on the full book against AUC where the money is still here",
            "A_full_exit · H=6 · mean over rolling origins", ylab="AUC",
            fmt=lambda v: f"{v:.3f}", save="v10c_auc_bars", ylim=(0.5, 0.95),
            colors={"auc_full_book": GREY, "full_model_where_money_still_here": ACC2,
                    "retrained_where_money_still_here": ACC})


## 8 · Savings where the money still is

The same engine as v10, run on both risk sets. A save applies only to a client who was genuinely
leaving; on success the balance freezes at the first alert, so `bar_now` there is what is retained.

Read in this order:

1. **8a — α sweep, both risk sets side by side.** On the full book the combined models are dragged
   toward empty accounts. On the money-still-here set they are not. If the timing story is right,
   the ordering between `payment_only` and `all_features` should change between the panels (P7).
2. **8b — the headline**: feature set × save rate at α = 1, on the money-still-here set.
3. **8c — uplift over a deposit-only queue.** This is the answer to *was the payment work worth
   doing*, with the empty-account artefact removed.
4. **8e — ranking on current against normal balance** (`bar_now` against `bar_med12`) at α = 1.

In [ ]:
# =====================================================================
# 8 · SAVINGS                                            [OUTPUT BLOCK 9]
# =====================================================================
t0 = time.time()
SAV = pd.concat([savings_grid(SC[("A_full_exit", c)], c) for c in CONFIGS], ignore_index=True)
SAV.to_csv(OUT_DIR / "v10c_savings.csv", index=False)
print(f"  savings grid {len(SAV):,} rows in {time.time()-t0:,.0f}s")
_k = SAV[(SAV.k == QUEUE_K) & (SAV.p_save == P_SAVE_BASE)]

# (a) alpha sweep, both risk sets
SWEEP = {c: (_k[_k.cfg == c].pivot_table(index="alpha", columns="feature_set",
                                          values="dollars_reached")[FS_ORDER]) for c in CONFIGS}
disp(pd.concat({c: SWEEP[c].T for c in CONFIGS}, axis=0).apply(lambda s: s.map(usd)).reset_index()
     .rename(columns={"level_0": "risk_set"}),
     title=f"8a &middot; <b>Dollars reached by &alpha;</b>, {QUEUE_K:,} alerts a month. "
           f"<code>full</code> reproduces v10b under the IPW fix; <code>defend</code> removes the "
           f"accounts whose money has already gone", n=10, save="v10c_alpha_sweep")
if HAVE_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
    for ax, c, ttl in zip(axes, CONFIGS, ["full book", "money still here"]):
        for fs in FS_ORDER:
            ax.plot(SWEEP[c].index, SWEEP[c][fs], marker="o", ms=4.5, lw=2,
                    color=FSCOL[fs], label=fs)
        ax.set_title(ttl, fontsize=9.5); ax.set_xlabel("α")
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    axes[0].set_ylabel("defendable $ reached"); axes[0].legend(frameon=False, fontsize=8)
    _fin(fig, "8a · Dollars reached by α, on the full book and where the money is still here",
         f"{QUEUE_K:,} alerts/month · A_full_exit · IPW + isotonic", save="v10c_alpha_panels")
RES["def_a1_all"] = float(SWEEP["defend"].loc[1.0, "all_features"])
RES["def_a1_pay"] = float(SWEEP["defend"].loc[1.0, "payment_only"])
ASTAR = {c: {fs: float(SWEEP[c][fs].idxmax()) for fs in FS_ORDER} for c in CONFIGS}

# (b) THE HEADLINE — alpha = 1, money still here
_d1 = SAV[(SAV.cfg == "defend") & (SAV.k == QUEUE_K) & (SAV.alpha == 1.0)]
H1 = _d1.pivot_table(index="feature_set", columns="p_save",
                     values="saved_annualised").reindex(FS_ORDER)
disp(H1.apply(lambda s: s.map(usd)).reset_index(),
     title=f"8b &middot; <b>Annualised dollars retained — money still here, &alpha; = 1.</b> "
           f"{QUEUE_K:,} alerts a month; success counted only on genuine attriters; balance "
           f"frozen at the first alert", save="v10c_headline")
heat(H1, "8b · Annualised dollars retained, by feature set and RM save rate",
     f"money-still-here risk set · α = 1 · {QUEUE_K:,} alerts/month · A_full_exit",
     xlab="RM save rate", save="v10c_heat_headline")
bar_grouped(H1.T, "RM save rate", FS_ORDER,
            "8b · Retained dollars by feature set, at every save rate",
            f"money-still-here · α = 1 · {QUEUE_K:,} alerts/month · annualised",
            ylab="annualised $ retained", colors=FSCOL, save="v10c_bar_headline")

# (c) uplift over deposit-only
UP = pd.DataFrame({f"{fs} over deposit_only": H1.loc[fs] - H1.loc["deposit_only"]
                   for fs in FS_ORDER[1:]})
disp(UP.apply(lambda s: s.map(usd)).reset_index().rename(columns={"index": "p_save"}),
     title="8c &middot; <b>What the payment work adds over a deposit-only queue</b>, annualised — "
           "same clients, folds, alert volume and freeze assumption; the only difference is what "
           "the model may see", save="v10c_uplift")

# (d) each set at its own best alpha, both risk sets
rows = []
for c in CONFIGS:
    for fs in FS_ORDER:
        r = SAV[(SAV.cfg == c) & (SAV.feature_set == fs) & (SAV.k == QUEUE_K) &
                (SAV.alpha == ASTAR[c][fs]) & (SAV.p_save == 0.01)].iloc[0]
        rows.append(dict(risk_set=c, feature_set=fs, alpha_star=ASTAR[c][fs],
                         departing_clients=int(r.tp_clients),
                         dollars_reached=usd(r.dollars_reached),
                         retained_at_1pct=usd(r.saved_annualised),
                         breakeven_save_rate=pctf(r.breakeven_p_save, 3)))
disp(pd.DataFrame(rows), title="8d &middot; Each feature set at <b>its own</b> best &alpha;, "
     "on both risk sets, at a 1% save rate", n=10, save="v10c_best_alpha")

# (e) rank on what the client holds now, not what it normally holds
rows = []
for c in CONFIGS:
    for fs in FS_ORDER:
        for bc in ["bar_med12", "bar_now"]:
            _, first = first_alerts(SC[("A_full_exit", c)], f"p_{fs}", QUEUE_K, 1.0, bc)
            tp = first[first.y == 1]
            rows.append(dict(risk_set=c, feature_set=fs, rank_on=bc,
                             dollars_reached=float(tp.bar_now.sum()), departing_clients=len(tp)))
VC = pd.DataFrame(rows)
VCP = VC.pivot_table(index=["risk_set", "feature_set"], columns="rank_on",
                     values="dollars_reached").reset_index()
VCP["gain_from_bar_now"] = VCP.bar_now/VCP.bar_med12.where(VCP.bar_med12 > 0) - 1
disp(VCP.assign(bar_med12=VCP.bar_med12.map(usd), bar_now=VCP.bar_now.map(usd),
                gain_from_bar_now=VCP.gain_from_bar_now.map(pctf)),
     title="8e &middot; <b>Rank on normal balance or current balance?</b> At &alpha; = 1. A save "
           "retains the current balance, so ranking on it should reach more — unless current "
           "balances are too noisy to rank on", n=10, save="v10c_rank_on")

# (f) the case at the most pessimistic save rate
_r = lambda fs: SAV[(SAV.cfg == "defend") & (SAV.feature_set == fs) & (SAV.k == QUEUE_K) &
                    (SAV.alpha == 1.0) & (SAV.p_save == 0.01)].iloc[0]
_A, _P, _D = _r("all_features"), _r("payment_only"), _r("deposit_only")
kv([("risk set", "money still here — at least half the normal balance, and at least $10k"),
    ("alert volume", f"{QUEUE_K:,} a month"), ("save rate", "1% — the most pessimistic tested"),
    ("all_features · departing clients reached", f"{int(_A.tp_clients):,}"),
    ("all_features · defendable dollars", usd(_A.dollars_reached)),
    ("all_features · retained, annualised", usd(_A.saved_annualised)),
    ("all_features · RM cost, annualised", usd(_A.rm_cost_annualised)),
    ("all_features · net, annualised", usd(_A.net_annualised)),
    ("all_features · break-even save rate", pctf(_A.breakeven_p_save, 3)),
    ("payment_only · retained, annualised", usd(_P.saved_annualised)),
    ("deposit_only · retained, annualised", usd(_D.saved_annualised)),
    ("uplift of all_features over deposit_only", usd(_A.saved_annualised - _D.saved_annualised)),
    ("does all_features now reach at least payment_only?",
     "yes" if _A.dollars_reached >= _P.dollars_reached else "no")],
   title="8f &middot; <b>The case at a 1% save rate, where the money is still here</b>",
   save="v10c_case")

# (g) the drained-but-open population: A only against A-or-B
_ab = savings_grid(SC[("AB_any", "defend")], "defend", alphas=[1.0], caps=[QUEUE_K])
_a1 = SAV[(SAV.cfg == "defend") & (SAV.alpha == 1.0) & (SAV.k == QUEUE_K)]
AB = pd.DataFrame({"A only": _a1[_a1.p_save == 0.01].set_index("feature_set").saved_annualised,
                   "A or B": _ab[_ab.p_save == 0.01].set_index("feature_set").saved_annualised}
                  ).reindex(FS_ORDER)
disp(AB.apply(lambda s: s.map(usd)).reset_index(),
     title="8g &middot; <b>Counting clients who drain without closing.</b> Annualised at a 1% save "
           "rate, money still here, &alpha; = 1. Nobody alerts on B clients today, so the "
           "difference is pure addition", save="v10c_AB")
bar_grouped(AB, "feature set", ["A only", "A or B"],
            "8g · Adding the drained-but-open population", "money still here · α = 1 · 1% save",
            ylab="annualised $ retained", save="v10c_bar_AB",
            colors={"A only": GREY, "A or B": ACC})


In [ ]:
# =====================================================================
# 9 · HOW MANY CALLS — the efficient frontier, fixed      [OUTPUT BLOCK 10]
# =====================================================================
rows = []
for fs in FS_ORDER:
    d = SAV[(SAV.cfg == "defend") & (SAV.feature_set == fs) & (SAV.alpha == 1.0) &
            (SAV.p_save == P_SAVE_BASE)][["k", "conversations", "dollars_reached"]]
    f = frontier(d); f["feature_set"] = fs
    for ps in [0.01, 0.10]:
        # per conversation, so no annualisation: numerator and denominator
        # cover the same test window
        f[f"retained_per_extra_call_{int(round(ps*100))}pct"] = f.marginal_per_conversation*ps
    rows.append(f)
FR = pd.concat(rows, ignore_index=True)
FR = FR[FR.k > 0]
disp(FR.assign(dollars_reached=FR.dollars_reached.map(usd),
               marginal_per_conversation=FR.marginal_per_conversation.map(usd),
               retained_per_extra_call_1pct=FR.retained_per_extra_call_1pct.map(usd),
               retained_per_extra_call_10pct=FR.retained_per_extra_call_10pct.map(usd)),
     title=f"9a &middot; <b>Efficient frontier.</b> Each row is a capacity that adds money; "
           f"<code>retained_per_extra_call</code> is what one more conversation keeps at that save "
           f"rate. <b>Stop where it falls below {usd(RM_COST_PER_CALL)}</b>, the cost of a call",
     n=40, save="v10c_frontier")
stop = {}
for fs in FS_ORDER:
    f = FR[FR.feature_set == fs]
    for ps, col in [(0.01, "retained_per_extra_call_1pct"), (0.10, "retained_per_extra_call_10pct")]:
        ok = f[f[col] >= RM_COST_PER_CALL]
        stop[(fs, ps)] = int(ok.k.max()) if len(ok) else 0
def _stop_txt(k):
    if k == 0: return "none — not worth a call even at the top of the list"
    if k == max(CAPACITY): return f"{k:,}  (the top of the tested grid — more calls still pay)"
    return f"{k:,}"
kv([(f"{fs} · largest worthwhile K at {int(round(ps*100))}% save", _stop_txt(stop[(fs, ps)]))
    for fs in FS_ORDER for ps in [0.01, 0.10]],
   title="9b &middot; How many alerts a month are worth making", save="v10c_stop")
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    for fs in FS_ORDER:
        f = FR[FR.feature_set == fs].dropna(subset=["retained_per_extra_call_1pct"])
        ax.plot(f.conversations, f.retained_per_extra_call_1pct, marker="o", ms=5, lw=2,
                color=FSCOL[fs], label=fs)
    ax.axhline(RM_COST_PER_CALL, color=INK, ls=":", lw=1.4)
    ax.text(ax.get_xlim()[1], RM_COST_PER_CALL, f"cost of a call {usd(RM_COST_PER_CALL)}  ",
            fontsize=8, va="bottom", ha="right")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("distinct conversations over the test window")
    ax.set_ylabel("$ retained by one more conversation, 1% save")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    ax.legend(frameon=False, fontsize=8)
    _fin(fig, "9 · How many calls? Stop where the curve crosses the cost line",
         "slopes of the upper concave envelope — decreasing by construction · 1% save rate",
         save="v10c_frontier_plot")


In [ ]:
# =====================================================================
# 10 · PERMUTATION — deposit_only, PERM_REPS per fold     [OUTPUT BLOCK 11]
# =====================================================================
# v10b: mean 0.544 over 7 single draws, sd 0.065, flagged. With few features a
# fit on shuffled labels tends to lean on one direction, so single draws swing
# widely. Twenty per fold gives the null distribution rather than one sample.
rng = np.random.default_rng(SEED)
_cols = [c for c in FSETS["deposit_only"] if c in TRw.columns]
rows = []; t0 = time.time()
for T in ORIGINS:
    trf = label(TRw[TRw.m_idx <= T - PRIMARY_H], "A_full_exit", PRIMARY_H)
    te = label(TE[T], "A_full_exit", PRIMARY_H)
    if trf.y.sum() < MIN_TRAIN_POS or te.y.sum() < 1: continue
    yte = te.y.to_numpy()
    real = auc(yte, predict_p(fit_spec(trf, _cols), te))
    for r in range(PERM_REPS):
        sh = trf.assign(y=rng.permutation(trf.y.to_numpy()))
        rows.append(dict(origin=T, rep=r, auc_real=real,
                         auc_perm=auc(yte, predict_p(fit_spec(sh, _cols), te))))
PERM = pd.DataFrame(rows)
_fm = PERM.groupby("origin").auc_perm.mean()
_m, _se = float(PERM.auc_perm.mean()), float(_fm.std(ddof=1)/np.sqrt(len(_fm)))
RES["perm_ci"] = (_m - 1.96*_se, _m + 1.96*_se)
kv([("fits", f"{len(PERM):,} permuted + {PERM.origin.nunique()} real, "
             f"{time.time()-t0:,.0f}s"),
    ("real AUC, mean over folds", f"{PERM.groupby('origin').auc_real.first().mean():.4f}"),
    ("permuted AUC, mean", f"{_m:.4f}"),
    ("95% interval of the mean (fold-level)", f"{RES['perm_ci'][0]:.4f} – {RES['perm_ci'][1]:.4f}"),
    ("single-draw range", f"{PERM.auc_perm.min():.3f} – {PERM.auc_perm.max():.3f}"),
    ("v10b's flagged value", "0.544 (7 single draws)"),
    ("verdict", "noise — the interval covers 0.50" if RES["perm_ci"][0] <= 0.5 <= RES["perm_ci"][1]
                else "NOT covered — investigate before trusting deposit_only")],
   title=f"10 &middot; <b>Permutation control, {PERM_REPS} per fold.</b> Shuffled training labels "
         f"must give chance performance on the untouched test month", save="v10c_permutation")
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    ax.hist(PERM.auc_perm, bins=30, color=GREY, alpha=.85)
    ax.axvline(0.5, color=INK, ls=":", lw=1.4, label="chance")
    ax.axvline(_m, color=ACC, lw=2, label=f"mean {_m:.3f}")
    ax.axvline(0.544, color=WARN, lw=1.4, ls="--", label="v10b flag 0.544")
    ax.set_xlabel("AUC with shuffled training labels"); ax.set_ylabel("fits")
    ax.legend(frameon=False, fontsize=8)
    _fin(fig, "10 · The permutation null for deposit_only",
         f"{len(PERM):,} fits across {PERM.origin.nunique()} folds", save="v10c_perm_hist")


In [ ]:
# =====================================================================
# 11 · THE POOL WITH B + THE LEAD LADDER PER CLIENT       [OUTPUT BLOCK 12]
# =====================================================================
att_A = lab.filter(F.col("q_A_full_exit").isNotNull()).select(
    "cust_pwr_id", F.col("q_A_full_exit").alias("event_m"))
att_B = (lab.filter(F.col("q_B_bal_exit").isNotNull() & F.col("q_A_full_exit").isNull())
         .select("cust_pwr_id", F.col("q_B_bal_exit").alias("event_m")))
def pool(df, nm):
    r = (BAL.join(df, "cust_pwr_id", "inner")
         .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
         .filter(F.col("rel_m") == -12)
         .agg(F.count(F.lit(1)).alias("n"), F.sum("bal_med12").alias("pool"),
              F.expr("percentile_approx(bal_med12, 0.5)").alias("med")).collect()[0])
    n_, p_ = int(r["n"]), float(r["pool"] or 0.0)
    return dict(cohort=nm, clients=n_, pool=p_, mean_relationship=p_/max(n_, 1),
                median_relationship=float(r["med"] or 0.0))
# column names avoid DataFrame method names — v10b died on PB.mean
PB = pd.DataFrame([pool(att_A, "A_full_exit"), pool(att_B, "B_bal_exit only")])
PB.loc[len(PB)] = dict(cohort="combined", clients=int(PB.clients.sum()), pool=PB["pool"].sum(),
                       mean_relationship=PB["pool"].sum()/max(PB.clients.sum(), 1),
                       median_relationship=np.nan)
disp(PB.assign(pool=PB["pool"].map(usd), mean_relationship=PB["mean_relationship"].map(usd),
               median_relationship=PB["median_relationship"].map(usd)),
     title="11a &middot; <b>The pool, with drained-but-open clients included</b> "
           "(relationship size at 12 months before the event)", save="v10c_pool")
kv([("A pool", usd(PB.loc[0, "pool"])), ("B-only pool", usd(PB.loc[1, "pool"])),
    ("combined", usd(PB.loc[2, "pool"])),
    ("mean relationship — A", usd(PB.loc[0, "mean_relationship"])),
    ("mean relationship — B-only", usd(PB.loc[1, "mean_relationship"])),
    ("B-only pool as a share of the A pool", pctf(PB.loc[1, "pool"]/max(PB.loc[0, "pool"], 1), 0))],
   title="11b &middot; What the business case has been leaving out", save="v10c_pool_verdict")

# lead ladder — the clients the headline queue catches, reached L months earlier
_, first = first_alerts(SC[("A_full_exit", "defend")], "p_all_features", QUEUE_K, 1.0)
tp = first[first.y == 1]
caught = spark.createDataFrame([(str(a), int(b)) for a, b in zip(tp.cust_pwr_id, tp.event_m)],
                               ["cust_pwr_id", "event_m"])
LD = collect_pd(BAL.select("cust_pwr_id", "m_idx", "bal_now")
                .join(F.broadcast(caught), "cust_pwr_id", "inner")
                .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
                .filter(F.col("rel_m").isin([-l for l in LEAD_GRID]))
                .groupBy("rel_m").agg(F.sum("bal_now").alias("dollars"),
                                      F.countDistinct("cust_pwr_id").alias("clients")),
                "lead ladder")
LD["lead_months"] = -LD.rel_m
LD["per_client"] = LD.dollars/LD.clients.clip(lower=1)
LD = LD.sort_values("lead_months").reset_index(drop=True)
_today = float((tp.event_m - tp.m_idx).median()) if len(tp) else np.nan
disp(LD.assign(dollars=LD.dollars.map(usd), per_client=LD.per_client.map(usd))
     [["lead_months", "clients", "dollars", "per_client"]],
     title=f"11c &middot; <b>What the same caught clients held L months before their event.</b> "
           f"Median lead today {_today:.0f} months. <code>per_client</code> is the comparable "
           f"column — the raw sum falls at long leads because fewer clients have that much history",
     save="v10c_lead")
if HAVE_MPL and len(LD):
    fig, ax1 = plt.subplots(figsize=(9.5, 4.0))
    ax1.bar(LD.lead_months, LD.per_client, color=ACC2, alpha=.9)
    ax1.set_xlabel("months of lead"); ax1.set_ylabel("defendable $ per client")
    ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    ax2 = ax1.twinx(); ax2.grid(False)
    ax2.plot(LD.lead_months, LD.clients, marker="o", color=ACC, lw=2)
    ax2.set_ylabel("clients with that much history", color=ACC)
    _fin(fig, "11c · What earlier detection is worth, per client",
         "same caught clients · money-still-here queue · α = 1", save="v10c_lead_plot")
_pc = LD.set_index("lead_months").per_client
kv([(f"per client at {l} months' lead", usd(_pc.get(l, np.nan))) for l in LEAD_GRID] +
   [("multiple, 2 → 6 months", f"{_pc.get(6, np.nan)/max(_pc.get(2, np.nan), 1):.1f}x")],
   title="11d &middot; The lead-time premium", save="v10c_lead_verdict")


In [ ]:
# =====================================================================
# 12 · SCORECARD — the pre-registered predictions         [OUTPUT BLOCK 13]
# =====================================================================
def _chk(key, test, fmt):
    if key not in RES: return "not evaluated", "—"
    v = RES[key]
    try: return ("✓" if test(v) else "✗"), fmt(v)
    except Exception as e: return f"error: {type(e).__name__}", "—"
PRED = [
    ("P1", "Audit implies a v10b miscalibration factor of 1.7–2.5x", "~75%",
     "audit_factor", lambda v: 1.7 <= v <= 2.5, lambda v: f"{v:.2f}x"),
    ("P2", "IPW + isotonic top-decile ratio in 0.80–1.25 for all four sets", "~70%",
     "cal_top_range", lambda v: 0.80 <= v[0] and v[1] <= 1.25, lambda v: f"{v[0]:.2f}–{v[1]:.2f}"),
    ("P3", "deposit_only, α=0: over 80% of reached attriters already drained", "~75%",
     "drained_dep_a0", lambda v: v > 0.80, pctf),
    ("P4", "payment_only, α=0: under 50% already drained", "~65%",
     "drained_pay_a0", lambda v: v < 0.50, pctf),
    ("P5", "deposit_only AUC falls ≥ 0.05 where money is still here", "~65%",
     "auc_drop_dep", lambda v: v >= 0.05, lambda v: f"{v:+.4f}"),
    ("P6", "payment_only AUC falls < 0.03 on the same subset", "~55%",
     "auc_drop_pay", lambda v: v < 0.03, lambda v: f"{v:+.4f}"),
    ("P7", "Money still here, α=1: all_features reaches ≥ payment_only dollars", "~55%",
     "def_a1_all", lambda v: v >= RES["def_a1_pay"],
     lambda v: f"{usd(v)} vs {usd(RES.get('def_a1_pay'))}"),
    ("P8", "deposit_only permutation 95% interval covers 0.50", "~80%",
     "perm_ci", lambda v: v[0] <= 0.5 <= v[1], lambda v: f"{v[0]:.3f}–{v[1]:.3f}"),
]
rows = []
for pid, text, conf, key, test, fmt in PRED:
    res, obs = _chk(key, test, fmt)
    rows.append(dict(id=pid, prediction=text, confidence=conf, observed=obs, result=res))
SCORE = pd.DataFrame(rows)
disp(SCORE, title="12 &middot; <b>Scorecard.</b> Each prediction was written in the introduction "
     "before this notebook ran; the result column is computed, not typed", n=10,
     save="v10c_scorecard")
_h = int((SCORE.result == "✓").sum()); _m_ = int((SCORE.result == "✗").sum())
print(f"  {_h} hit · {_m_} missed · {len(SCORE)-_h-_m_} not evaluated")


---

## Reading this run

**If P3–P6 hit**, the timing story is confirmed and it is the cleanest slide in the programme:
*deposit balances find attriters after the money has gone; payment behaviour finds them while it
is still there.* 6b is the chart to show — the deposit-only histogram piled up left of the
"drained" line, the payment histogram spread across the live range.

**If P7 hits**, the combined model was never worse — it was being rewarded for the wrong thing.
Evaluated where the money is still here, `all_features` leads, and **8c is the value of the payment
work**, quoted at a 1% save rate so it cannot be argued down. If P7 misses, payment features
carry the queue on their own and the balance features add nothing where it matters; the product
should be built on them, with balance used for sizing (α) rather than as a predictor.

**Whichever way P7 goes**, the operating definition changes: **the queue should only ever contain
client-months where the money is still here.** Alerting on an empty account costs a call and
cannot retain anything.

**11d still sets the v11 agenda.** If defendable balance per client at 6 months' lead is several
times the figure at 2 months, the next release is about predicting earlier — `D_money_move`,
whose lead in the top balance decile was 5 months in v10b — not about seeing more.

### Still outstanding
No live evidence that an alert changes an outcome. The 20% of the deposit book invisible to
payments has never been profiled. And the save rate needs a pilot with a **holdout arm**: in this
data a save and a client who was never going to leave look identical, so no amount of back-testing
can measure it.